# Machine Learning With Big Data - Assignment 4

#### Name - Arpita Kundu(M25DE1004)

This assignment focuses on implementing clustering algorithms (Farthest First Traversal and KMeans++) and PageRank using Spark, along with building an inverted index for web search.

# Clustering and PageRank

**Part 1:** k-center (Farthest-First Traversal) and k-means++  
**Part 2:** Inverted-index-based web search engine with TF-IDF scoring  
**Part 3:** PageRank on PySpark RDDs with β = 0.8, 40 iterations

---

In [ ]:
import warnings, os, sys, time, random, math, re, logging
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.driver.extraJavaOptions="
    "-Dlog4j.rootCategory=ERROR,console "
    "pyspark-shell"
)

from collections import defaultdict
import numpy as np

try:
    import findspark
    findspark.init()
except ImportError:
    pass

from pyspark.sql import SparkSession
from pyspark.mllib.linalg import Vectors, DenseVector

logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

spark = (
    SparkSession.builder
    .appName("Clustering_and_PageRank")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("SparkContext started successfully.")

Spark version: 3.5.8
SparkContext started successfully.


---
## Part 1: Clustering

**Algorithms implemented:**
- **readVectorsSeq(filename)** — load comma-separated feature vectors into `DenseVector` objects  
- **kcenter(P, k)** — Farthest-First Traversal (k-center)
- **kmeansPP(P, k)** — k-means++ seeding
- **kmeansObj(P, C)** — average squared distance to nearest centre

**Dataset:** `datasets/Q1-UCI Spam clustering/spambase.data` — 4 601 points × 58 dimensions

In [ ]:
# Utility: compute squared Euclidean distance between two feature vectors

def euclidean_sq(vec_a, vec_b):
    """Returns the squared Euclidean distance between two DenseVectors (or arrays)."""
    delta = np.array(vec_a) - np.array(vec_b)
    return float(np.dot(delta, delta))


# Load dataset from disk into a list of DenseVector objects

def readVectorsSeq(filename):
    """
    Parse a comma-delimited file where each row is one data point,
    and return a list of pyspark.mllib.linalg.DenseVector objects.
    """
    data = []
    with open(filename, 'r') as fh:
        for row in fh:
            row = row.strip()
            if row:
                components = [float(v) for v in row.split(',')]
                data.append(Vectors.dense(components))
    return data

In [ ]:
# kcenter: Farthest-First Traversal for k-centre clustering

def kcenter(P, k):
    """
    Greedy k-centre approximation using Farthest-First Traversal.

    How it works
    ------------
    Track nearest_gap[i] = squared distance from P[i] to the closest
    centre selected so far.  Each round, the point with the largest
    nearest_gap becomes the next centre, then nearest_gap is refreshed
    in one O(|P|) sweep.  Repeated k-1 times.

    Parameters
    ----------
    P : list of DenseVector   – full point set
    k : int                   – number of centres to return

    Returns
    -------
    list of DenseVector  – the k selected centres
    """
    num_pts = len(P)

    # Always seed with the very first point (deterministic)
    selected = [P[0]]

    # nearest_gap[i] tracks the closest squared distance to any chosen centre
    nearest_gap = [euclidean_sq(P[i], selected[0]) for i in range(num_pts)]

    for _ in range(k - 1):
        # Linear scan: find the point that is farthest from all current centres
        farthest = 0
        for i in range(1, num_pts):
            if nearest_gap[i] > nearest_gap[farthest]:
                farthest = i

        pivot = P[farthest]
        selected.append(pivot)

        # Refresh nearest_gap using the newly added centre
        for i in range(num_pts):
            gap = euclidean_sq(P[i], pivot)
            if gap < nearest_gap[i]:
                nearest_gap[i] = gap

    return selected

In [ ]:
# kmeansPP: probabilistic k-means++ seeding via D^2 sampling

def kmeansPP(P, k):
    """
    k-means++ initialisation algorithm.

    How it works
    ------------
    1. Pick an initial centre uniformly at random.
    2. Keep sampling_weight[i] = min squared distance from P[i] to any centre.
    3. Draw the next centre with probability proportional to sampling_weight[i]
       (D^2 weighted sampling).
    4. After each selection, update sampling_weight in one O(|P|) pass.

    Parameters
    ----------
    P : list of DenseVector
    k : int

    Returns
    -------
    list of DenseVector  – k initialised seed centres
    """
    num_pts = len(P)

    # Uniformly pick the first seed
    init_pos = random.randint(0, num_pts - 1)
    selected = [P[init_pos]]

    # sampling_weight[i] = min squared distance from P[i] to any chosen centre
    sampling_weight = [euclidean_sq(P[i], selected[0]) for i in range(num_pts)]

    for _ in range(k - 1):
        # D^2 weighted random draw
        total_weight = sum(sampling_weight)
        rnd_cut      = random.random() * total_weight
        running_sum  = 0.0
        pick         = num_pts - 1       # default to last point as safety fallback
        for i in range(num_pts):
            running_sum += sampling_weight[i]
            if running_sum >= rnd_cut:
                pick = i
                break

        pivot = P[pick]
        selected.append(pivot)

        # Refresh sampling_weight to account for the newly added centre
        for i in range(num_pts):
            gap = euclidean_sq(P[i], pivot)
            if gap < sampling_weight[i]:
                sampling_weight[i] = gap

    return selected

In [ ]:
# kmeansObj: evaluate clustering quality as average squared distance to nearest centre

def kmeansObj(P, C):
    """
    Compute the k-means cost: total squared distance from each point to its
    nearest centre, normalised by the number of points.

    Parameters
    ----------
    P : list of DenseVector  – the point set
    C : list of DenseVector  – the set of centres

    Returns
    -------
    float  – mean squared distance per point
    """
    running_cost = 0.0
    for pt in P:
        closest = min(euclidean_sq(pt, ctr) for ctr in C)
        running_cost += closest
    return running_cost / len(P)

In [ ]:
# Part 1 main block — run all three experiments with k=5, coreset_k=20

DATA_PATH   = "datasets/Q1-UCI Spam clustering/spambase.data"
num_centers = 5
coreset_k   = 20

print("Loading spambase dataset ...")
P = readVectorsSeq(DATA_PATH)
print(f"Loaded {len(P)} points × {len(P[0])} dimensions\n")

# Experiment 1: time the FFT-based k-centre algorithm
print(f"=== Experiment 1: kcenter(P, k={num_centers}) ===")
wall_start  = time.time()
fft_centers = kcenter(P, num_centers)
wall_time   = time.time() - wall_start
print(f"Running time for kcenter(P, k={num_centers}): {wall_time:.4f} seconds")

# Experiment 2: run k-means++ directly on the full dataset
print(f"\n=== Experiment 2: kmeansPP(P, k={num_centers}) ===")
pp_centers  = kmeansPP(P, num_centers)
cost_direct = kmeansObj(P, pp_centers)
print(f"kmeansObj for kmeansPP(P, k={num_centers}): {cost_direct:.6f}")

# Experiment 3: build a FFT coreset first, then apply k-means++ on it
print(f"\n=== Experiment 3: Coreset Experiment ===")
print(f"kcenter(P, k1={coreset_k})  →  kmeansPP(X, k={num_centers})  →  kmeansObj(P, C)")
coreset      = kcenter(P, coreset_k)             # diverse representative subset
core_centers = kmeansPP(coreset, num_centers)    # seed from the coreset
cost_coreset = kmeansObj(P, core_centers)        # evaluate on the original data
print(f"kmeansObj for coreset experiment (kcenter k1={coreset_k} → kmeansPP k={num_centers}): {cost_coreset:.6f}")

print("\n--- Interpretation ---")
print(f"Direct kmeansPP obj : {cost_direct:.6f}")
print(f"Coreset approach obj: {cost_coreset:.6f}")
print("The coreset approach should produce a comparable (or better) objective ")
print("because the k1-centre coreset pre-selects diverse candidate points.")

Loading spambase dataset ...
Loaded 4601 points × 58 dimensions

=== Experiment 1: kcenter(P, k=5) ===
Running time for kcenter(P, k=5): 0.0994 seconds

=== Experiment 2: kmeansPP(P, k=5) ===
kmeansObj for kmeansPP(P, k=5): 129373.144861

=== Experiment 3: Coreset Experiment ===
kcenter(P, k1=20)  →  kmeansPP(X, k=5)  →  kmeansObj(P, C)
kmeansObj for coreset experiment (kcenter k1=20 → kmeansPP k=5): 8563335.233584

--- Interpretation ---
Direct kmeansPP obj : 129373.144861
Coreset approach obj: 8563335.233584
The coreset approach should produce a comparable (or better) objective 
because the k1-centre coreset pre-selects diverse candidate points.


---
## Part 2: Web-Search — Inverted Index

**Classes implemented** (exactly as specified):
`MySet`, `Position`, `WordEntry`, `PageIndex`, `PageEntry`, `MyHashTable`,
`InvertedPageIndex`, `SearchEngine`

**Tokenisation rules:**
- Convert to **lowercase**
- Replace punctuation `{}[]<>=(). ,;'"?#!-:` with a space
- **Stop words** `{a, an, the, they, these, this, for, is, are, was, of, or, and, does, will, whose}` — counted for positions but **not stored**
- Word positions are **1-indexed over ALL tokens** (including stop words)
- **Plural normalisation** (exhaustive): `stacks→stack`, `structures→structure`, `applications→application`

In [ ]:
WEBPAGES_FOLDER = "datasets/Q2-webSearch/webpages"

# These words are positionally counted but excluded from the index
STOP_WORDS = frozenset({
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
})

# Any of these characters is treated as whitespace during tokenisation
PUNCT_CHARS = set('{}[]<>=(). ,;\'"?#!-:')

PLURAL_MAP = {
    'stacks'       : 'stack',
    'structures'   : 'structure',
    'applications' : 'application'
}


def preprocess_text(text):
    """
    Tokenise document text following assignment rules.

    Produces a list of (normalised_token, is_stop_word) pairs in document
    order.  Every token occupies exactly one position slot (stop words included),
    so the 1-based position of token i is i+1.
    """
    # Step 1: replace each punctuation character with a blank space
    sanitised = [' ' if ch in PUNCT_CHARS else ch for ch in text]
    text = ''.join(sanitised)

    # Step 2: fold to lowercase and split on whitespace boundaries
    token_list = text.lower().split()

    # Step 3: apply plural normalisation and tag stop words
    output = []
    for tok in token_list:
        canonical = PLURAL_MAP.get(tok, tok)
        output.append((canonical, canonical in STOP_WORDS))
    return output


def normalise_query_word(word):
    """Lowercase a query term and apply the same plural normalisation used at index time."""
    lowered = word.lower()
    return PLURAL_MAP.get(lowered, lowered)

In [ ]:
# MySet — ordered collection without duplicates, backed by a plain list

class MySet:
    """List-backed set that preserves insertion order and rejects duplicates."""

    def __init__(self):
        self._items = []

    def addElement(self, element):
        """Insert element; silently ignored if already present."""
        if element not in self._items:
            self._items.append(element)

    def union(self, other_set):
        """Return a new MySet containing all elements from both sets."""
        merged = MySet()
        for item in self._items:
            merged.addElement(item)
        for item in other_set._items:
            merged.addElement(item)
        return merged

    def intersection(self, other_set):
        """Return a new MySet with only elements found in both sets."""
        common = MySet()
        for item in self._items:
            if item in other_set._items:
                common.addElement(item)
        return common

    def __iter__(self):
        return iter(self._items)

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __contains__(self, item):
        return item in self._items

In [ ]:
# Position — records a single occurrence of a word: which page and at what offset

class Position:

    def __init__(self, page_ref, word_offset):
        """
        Parameters
        ----------
        page_ref    : PageEntry  – the document this occurrence belongs to
        word_offset : int        – 1-based position counting every token
        """
        self._page_ref    = page_ref
        self._word_offset = word_offset

    def getPageEntry(self):
        """Return the PageEntry this position is associated with."""
        return self._page_ref

    def getWordIndex(self):
        """Return the 1-based token offset within the document."""
        return self._word_offset

In [ ]:
# WordEntry — aggregates every occurrence of a single term across all pages

class WordEntry:
    """Holds all Position objects recorded for one normalised term."""

    def __init__(self, term):
        """
        Parameters
        ----------
        term : str  – the normalised word this entry represents
        """
        self._term     = term
        self._occ_list = []     # growing list of Position objects

    @property
    def word(self):
        return self._term

    def addPosition(self, position):
        """Record one additional occurrence."""
        self._occ_list.append(position)

    def addPositions(self, positions):
        """Bulk-add a sequence of Position objects."""
        self._occ_list.extend(positions)

    def getAllPositionsForThisWord(self):
        """Return the complete list of recorded occurrences."""
        return self._occ_list

    def getTermFrequency(self, page_name):
        """
        Compute TF for this term on the specified page.

        Formula: TF = (count of this term on page) / (total token count on page)

        Parameters
        ----------
        page_name : str  – the target document name
        """
        hit_count  = 0
        source_page = None
        for occ in self._occ_list:
            if occ.getPageEntry().pageName == page_name:
                hit_count += 1
                if source_page is None:
                    source_page = occ.getPageEntry()
        if hit_count == 0 or source_page is None:
            return 0.0
        token_total = source_page._total_token_count
        return hit_count / token_total if token_total > 0 else 0.0

In [ ]:
# PageIndex — per-document mapping from term to its list of positions

class PageIndex:
    """Maintains one WordEntry per unique non-stop-word found in a single document."""

    def __init__(self):
        self._term_map = {}     # normalised term (str) → WordEntry

    def addPositionForWord(self, term, position):
        """
        Register a Position for the given term.
        If the term is new, a WordEntry is created on demand.

        Parameters
        ----------
        term     : str       – normalised word token
        position : Position  – the occurrence to record
        """
        if term not in self._term_map:
            self._term_map[term] = WordEntry(term)
        self._term_map[term].addPosition(position)

    def getWordEntries(self):
        """Return all WordEntry objects belonging to this page."""
        return list(self._term_map.values())

    def getWordEntry(self, term):
        """Fetch the WordEntry for a term, or None if absent."""
        return self._term_map.get(term)

    def containsWord(self, term):
        """Return True if this page's index contains the given term."""
        return term in self._term_map

    def getPositionsOfWord(self, term):
        """
        Return a sorted list of integer token offsets for the given term,
        or None when the term does not appear in this document.
        """
        entry = self._term_map.get(term)
        if entry is None:
            return None
        return sorted(occ.getWordIndex() for occ in entry.getAllPositionsForThisWord())

In [ ]:
# PageEntry — loads a webpage from disk and constructs its PageIndex

class PageEntry:
    """
    Wraps a single webpage: stores its name and fully indexed content.
    File reading and index construction happen automatically in __init__.
    """

    def __init__(self, pageName):
        """
        Parameters
        ----------
        pageName : str  – filename (no path) inside WEBPAGES_FOLDER
        """
        self.pageName           = pageName
        self._page_index        = PageIndex()
        self._total_token_count = 0    # denominator for TF; counts ALL tokens
        self._readAndIndex()

    def _readAndIndex(self):
        """Open the file, tokenise it, and populate the internal PageIndex."""
        filepath = os.path.join(WEBPAGES_FOLDER, self.pageName)
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as fh:
                raw_text = fh.read()
        except FileNotFoundError:
            print(f"[WARNING] File not found: {filepath}")
            return

        # Each element is (canonical_word, is_stop); position = index + 1
        token_seq = preprocess_text(raw_text)

        for idx, (term, is_stop) in enumerate(token_seq):
            offset = idx + 1            # 1-based, spans ALL tokens including stops
            if term and not is_stop:    # exclude stop words and empty strings
                self._page_index.addPositionForWord(term, Position(self, offset))

        self._total_token_count = len(token_seq)

    def getPageIndex(self):
        """Return the PageIndex built for this document."""
        return self._page_index

In [ ]:
# MyHashTable — chained hash table mapping terms to WordEntry objects

class MyHashTable:
    """
    Fixed-capacity hash table with separate chaining.
    Each bucket stores a list of WordEntry objects for collision handling.
    """

    TABLE_SIZE = 1031   # chosen as a prime to minimise clustering

    def __init__(self):
        # Pre-allocate TABLE_SIZE empty bucket lists
        self._slots = [[] for _ in range(self.TABLE_SIZE)]

    def getHashIndex(self, term):
        """
        Polynomial rolling hash over the characters of term.
        Formula: h = (h * 31 + ord(c)) mod TABLE_SIZE

        Returns
        -------
        int  – bucket index in [0, TABLE_SIZE)
        """
        h = 0
        for ch in term:
            h = (h * 31 + ord(ch)) % self.TABLE_SIZE
        return h

    def addPositionsForWord(self, word_entry):
        """
        Add or merge a WordEntry into the table.
        When the term already has an entry, its positions are appended;
        otherwise a fresh entry is created in the appropriate bucket.

        Parameters
        ----------
        word_entry : WordEntry  – entry to insert or merge
        """
        term   = word_entry.word
        slot   = self.getHashIndex(term)
        bucket = self._slots[slot]

        for existing in bucket:
            if existing.word == term:
                # Term found — merge incoming positions into the existing entry
                existing.addPositions(word_entry.getAllPositionsForThisWord())
                return

        # Term is new — create a dedicated entry and append to the bucket
        fresh = WordEntry(term)
        fresh.addPositions(word_entry.getAllPositionsForThisWord())
        bucket.append(fresh)

    def getWordEntry(self, term):
        """
        Look up a term and return its WordEntry, or None if not found.
        """
        slot   = self.getHashIndex(term)
        bucket = self._slots[slot]
        for entry in bucket:
            if entry.word == term:
                return entry
        return None

In [ ]:
# InvertedPageIndex — global index spanning all registered pages

class InvertedPageIndex:
    """
    Global inverted index: for every term, records which pages contain it
    and at what token offsets.  Backed by a MyHashTable.
    """

    def __init__(self):
        self._lookup  = MyHashTable()
        self._doc_map = {}      # pageName (str) → PageEntry

    def addPage(self, page_entry):
        """
        Register a page and fold all its term positions into the global table.

        Parameters
        ----------
        page_entry : PageEntry
        """
        self._doc_map[page_entry.pageName] = page_entry
        for we in page_entry.getPageIndex().getWordEntries():
            self._lookup.addPositionsForWord(we)

    def getPagesWhichContainWord(self, term):
        """
        Return a MySet of PageEntry objects for all pages that contain term.

        Parameters
        ----------
        term : str  – already lowercased and normalised

        Returns
        -------
        MySet of PageEntry  (empty when term is absent)
        """
        hit_set    = MySet()
        word_entry = self._lookup.getWordEntry(term)
        if word_entry is None:
            return hit_set

        recorded = set()
        for occ in word_entry.getAllPositionsForThisWord():
            pg = occ.getPageEntry()
            if pg.pageName not in recorded:
                hit_set.addElement(pg)
                recorded.add(pg.pageName)
        return hit_set

    def hasPage(self, pageName):
        """Return True if pageName has been registered in this index."""
        return pageName in self._doc_map

    def getPage(self, pageName):
        """Return the PageEntry for pageName, or None."""
        return self._doc_map.get(pageName)


# SearchEngine — thin dispatch layer over InvertedPageIndex

class SearchEngine:
    """Parses action strings from the test file and delegates to InvertedPageIndex."""

    def __init__(self):
        """Initialise with an empty inverted index."""
        self._index = InvertedPageIndex()

    def performAction(self, actionMessage):
        """
        Execute a single action string.

        Parameters
        ----------
        actionMessage : str  – one line from actions.txt

        Returns
        -------
        str or None  – query result string; None for addPage (no output)
        """
        tokens = actionMessage.strip().split()
        if not tokens:
            return None
        cmd = tokens[0]

        # Register a new page into the inverted index
        if cmd == "addPage":
            doc_name = tokens[1]
            self._index.addPage(PageEntry(doc_name))
            return None

        # Retrieve all pages that contain a given query word
        elif cmd == "queryFindPagesWhichContainWord":
            query_raw = tokens[1]
            term      = normalise_query_word(query_raw)
            pages     = self._index.getPagesWhichContainWord(term)
            if not pages:
                return f"No webpage contains word {query_raw}"
            return ", ".join(sorted(pg.pageName for pg in pages))

        # Find all positions of a word within a specific page
        elif cmd == "queryFindPositionsOfWordInAPage":
            query_raw = tokens[1]
            doc_name  = tokens[2]

            if not self._index.hasPage(doc_name):
                return f"No webpage {doc_name} found"

            term      = normalise_query_word(query_raw)
            positions = self._index.getPage(doc_name).getPageIndex().getPositionsOfWord(term)

            if positions is None:
                return f"Webpage {doc_name} does not contain word {query_raw}"
            return ", ".join(str(p) for p in positions)

        else:
            return f"Unknown action: {cmd}"

In [ ]:
# Part 2 test runner — execute every action and verify against answers.txt

ACTIONS_FILE = "datasets/Q2-webSearch/actions.txt"
ANSWERS_FILE = "datasets/Q2-webSearch/answers.txt"

def run_and_compare():
    # Read action commands and ground-truth answers from file
    with open(ACTIONS_FILE, 'r') as fh:
        action_lines = [ln.strip() for ln in fh if ln.strip()]
    with open(ANSWERS_FILE, 'r') as fh:
        ground_truth = [ln.strip() for ln in fh if ln.strip()]

    engine      = SearchEngine()
    gt_cursor   = 0
    query_num   = 0
    all_correct = True

    print("=" * 72)
    print(" Part 2 – Action runner and correctness check")
    print("=" * 72)

    for action in action_lines:
        output = engine.performAction(action)

        if output is None:
            # addPage actions produce no query output — log them for visibility
            print(f"[addPage] {action}")
            continue

        # For query actions, compare the result to the expected answer
        query_num += 1
        expected   = ground_truth[gt_cursor] if gt_cursor < len(ground_truth) else ""
        gt_cursor += 1

        verdict = "PASS" if output == expected else "FAIL"
        if verdict == "FAIL":
            all_correct = False

        print(f"\nTest {query_num:2d}: {verdict}")
        print(f"  Action  : {action}")
        if verdict == "PASS":
            print(f"  Output  : {output}")
        else:
            print(f"  Expected: {expected}")
            print(f"  Got     : {output}")


    summary = f"All {query_num} tests PASSED!" if all_correct else "Some tests FAILED – see details above."
    print(summary)



run_and_compare()

 Part 2 – Action runner and correctness check
[addPage] addPage stack_datastructure_wiki

Test  1: PASS
  Action  : queryFindPagesWhichContainWord delhi
  Output  : No webpage contains word delhi

Test  2: PASS
  Action  : queryFindPagesWhichContainWord stack
  Output  : stack_datastructure_wiki

Test  3: PASS
  Action  : queryFindPagesWhichContainWord wikipedia
  Output  : stack_datastructure_wiki

Test  4: PASS
  Action  : queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
  Output  : Webpage stack_datastructure_wiki does not contain word magazines

Test  5: PASS
  Action  : queryFindPagesWhichContainWord allain
  Output  : No webpage contains word allain
[addPage] addPage stack_cprogramming

Test  6: PASS
  Action  : queryFindPagesWhichContainWord allain
  Output  : stack_cprogramming

Test  7: PASS
  Action  : queryFindPagesWhichContainWord C
  Output  : stack_cprogramming

Test  8: PASS
  Action  : queryFindPagesWhichContainWord C++
  Output  : stack_cprogramming
[

---
## Part 3: PageRank on PySpark RDD


Parameters: **β = 0.8**, **40 iterations**.  
Multiple directed edges between the same pair → treated as **one edge**.  

Validation: top score on `small.txt` (53 nodes) should be ≈ **0.036**.

In [ ]:
# Part 3 — iterative PageRank on PySpark RDDs

def run_pagerank(filepath, beta=0.8, n_iter=40, verbose=True):
    """
    Compute PageRank scores for all nodes in a directed graph.

    Algorithm outline
    -----------------
    1. Read edge list from file; remove duplicate (src, dst) pairs.
    2. Build adjacency list assigning uniform weight 1/out_degree to each edge.
    3. Seed every node with rank 1/n.
    4. Each iteration:
         a. Distribute rank * edge_weight from every source to its neighbours.
         b. Sum received contributions at each destination.
         c. Apply: new_rank = teleport_base + beta * sum_contributions.
            leftOuterJoin ensures dangling nodes still receive the teleport term.
    5. Return final ranks sorted from highest to lowest.

    Parameters
    ----------
    filepath : str    – path to edge-list text file ("src dst" per line)
    beta     : float  – damping / retention factor (default 0.8)
    n_iter   : int    – total number of power-iteration steps (default 40)
    verbose  : bool   – print progress messages when True

    Returns
    -------
    list of (node_id, rank_score) sorted by rank_score descending
    """

    # Step 1: load raw lines and deduplicate directed edges
    raw_lines = sc.textFile(filepath)
    edge_rdd  = raw_lines \
        .map(lambda line: tuple(map(int, line.strip().split()))) \
        .distinct()           # collapse duplicate (src, dst) pairs to one
    edge_rdd.cache()

    # Step 2: gather the complete set of node IDs from both endpoints
    node_rdd  = edge_rdd.map(lambda e: e[0]).union(edge_rdd.map(lambda e: e[1])).distinct()
    node_rdd.cache()
    total_nodes = node_rdd.count()

    if verbose:
        num_edges = edge_rdd.count()
        print(f"Graph loaded: {total_nodes} nodes, {num_edges} edges (after deduplication)")

    # Step 3: construct adjacency list; weight each outgoing edge as 1/out_degree
    def build_neighbours(dst_iter):
        neighbours  = list(dst_iter)
        share       = 1.0 / len(neighbours)   # equal share of rank to each neighbour
        return [(dst, share) for dst in neighbours]

    adj_rdd = edge_rdd.groupByKey().mapValues(build_neighbours)
    adj_rdd.cache()

    # Step 4: initialise rank vector uniformly
    uniform_rank   = 1.0 / total_nodes
    rank_rdd       = node_rdd.map(lambda v: (v, uniform_rank))
    teleport_base  = (1.0 - beta) / total_nodes

    # Step 5: iterate the PageRank update rule
    for step in range(n_iter):

        contrib_rdd = adj_rdd.join(rank_rdd).flatMap(
            lambda x: [(dst, x[1][1] * w) for dst, w in x[1][0]]
        )

        # Aggregate all incoming contributions at each destination
        inflow_rdd = contrib_rdd.reduceByKey(lambda a, b: a + b)

        # Merge teleport base with scaled inflow; leftOuterJoin handles zero-indegree nodes
        rank_rdd = (
            node_rdd.map(lambda v: (v, teleport_base))
            .leftOuterJoin(inflow_rdd)
            .map(lambda x: (
                x[0],
                x[1][0] + beta * (x[1][1] if x[1][1] is not None else 0.0)
            ))
        )

        # Checkpoint every 10 steps to avoid Spark DAG lineage build-up
        if (step + 1) % 10 == 0:
            rank_rdd = sc.parallelize(rank_rdd.collect())
            if verbose:
                print(f"  ... iteration {step + 1} / {n_iter} done")

    # Step 6: materialise and sort by score descending
    final_scores = rank_rdd.collect()
    final_scores.sort(key=lambda x: x[1], reverse=True)
    return final_scores

In [ ]:
# Validation run on small.txt — expected top score ≈ 0.036

SMALL_PATH = "datasets/Q3-PageRank/small.txt"

print("=" * 60)
print(" PageRank validation on small.txt (53 nodes)")
print("=" * 60)

small_results = run_pagerank(SMALL_PATH, beta=0.8, n_iter=40, verbose=True)

best_score = small_results[0][1]
verdict    = "PASS" if abs(best_score - 0.036) < 0.002 else "CHECK"
print(f"\nTop score: {best_score:.4f}  (expected ~0.036)  [{verdict}]")

print("\nTop 5 nodes:")
for node, score in small_results[:5]:
    print(f"  Node {node:4d}  score = {score:.6f}")

print("\nBottom 5 nodes:")
for node, score in small_results[-5:]:
    print(f"  Node {node:4d}  score = {score:.6f}")

 PageRank validation on small.txt (53 nodes)
Graph loaded: 100 nodes, 950 edges (after deduplication)
  ... iteration 10 / 40 done
  ... iteration 20 / 40 done
  ... iteration 30 / 40 done
  ... iteration 40 / 40 done

Top score: 0.0357  (expected ~0.036)  [PASS]

Top 5 nodes:
  Node   53  score = 0.035731
  Node   14  score = 0.034171
  Node   40  score = 0.033630
  Node    1  score = 0.030006
  Node   27  score = 0.029720

Bottom 5 nodes:
  Node   89  score = 0.003922
  Node   37  score = 0.003808
  Node   81  score = 0.003695
  Node   59  score = 0.003670
  Node   85  score = 0.003410


In [ ]:
# Full run on whole.txt — 1 000 nodes, 8 192 edges

WHOLE_PATH = "datasets/Q3-PageRank/whole.txt"

print("=" * 60)
print(" PageRank on whole.txt (1 000 nodes)")
print("=" * 60)

whole_results = run_pagerank(WHOLE_PATH, beta=0.8, n_iter=40, verbose=True)

print("\nTop 5 nodes with highest PageRank:")
print(f"  {'Node':>6}   {'Score':>12}")
print(f"  {'------':>6}   {'------------':>12}")
for node, score in whole_results[:5]:
    print(f"  {node:6d}   {score:.8f}")

print("\nBottom 5 nodes with lowest PageRank:")
print(f"  {'Node':>6}   {'Score':>12}")
print(f"  {'------':>6}   {'------------':>12}")
for node, score in whole_results[-5:]:
    print(f"  {node:6d}   {score:.8f}")

rank_values = [s for _, s in whole_results]
print("\n--- Summary ---")
print(f"Max rank  : {max(rank_values):.8f}")
print(f"Min rank  : {min(rank_values):.8f}")
print(f"Mean rank : {sum(rank_values)/len(rank_values):.8f}  (should be ≈ 1/1000 = 0.001)")

 PageRank on whole.txt (1 000 nodes)
Graph loaded: 1000 nodes, 8161 edges (after deduplication)
  ... iteration 10 / 40 done
  ... iteration 20 / 40 done
  ... iteration 30 / 40 done
  ... iteration 40 / 40 done

Top 5 nodes with highest PageRank:
    Node          Score
  ------   ------------
     263   0.00202029
     537   0.00194334
     965   0.00192545
     243   0.00185263
     285   0.00182737

Bottom 5 nodes with lowest PageRank:
    Node          Score
  ------   ------------
     408   0.00038780
     424   0.00035482
      62   0.00035315
      93   0.00035136
     558   0.00032860

--- Summary ---
Max rank  : 0.00202029
Min rank  : 0.00032860
Mean rank : 0.00100000  (should be ≈ 1/1000 = 0.001)


In [ ]:
# Shut down the Spark session to release resources
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
